## Data validation
- Validate data -> Save at validation folder
    - Data Type not match -> set schema table at config file and prepare init table
    - Not null columns (key) -> set not null at config file
    - Key duplicate -> set key at config file
    - Row duplicate -> set row uniqueness validation in config file

In [1]:
import os
os.chdir("../")

In [2]:
os.getcwd()

'/Users/supawitjunsiritrakhoon/Desktop/Customer_Churn_Prediction/Project_file/customer-churn-prediction'

### Import library

In [3]:
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional
from dataclasses import dataclass
import re
from datetime import datetime
from src.churn_prediction.logger import logger
from src.churn_prediction.pydantic.data_validation_config import DataValidationConfig
from src.churn_prediction.pydantic.pipeline_config import PipelineConfig
from src.churn_prediction.utils.common import load_single_config, generate_sk_key
from src.churn_prediction.utils.loaders import load_data
from src.churn_prediction.utils.writers import save_data

In [38]:
admin_group_df = load_data(source="data/user_coop_anonymized.csv", delimiter="|")
admin_group_df['admin_id'] = admin_group_df['admin_id'].astype(str)

[ 2025-11-08 13:01:21 ] | churn_prediction | INFO     | loaders.py:load_data:343 | Auto-detecting format for local file: data/user_coop_anonymized.csv
[ 2025-11-08 13:01:21 ] | churn_prediction | INFO     | loaders.py:load_csv:239 | Loading CSV from local: data/user_coop_anonymized.csv
[ 2025-11-08 13:01:21 ] | churn_prediction | INFO     | loaders.py:load_csv:241 | ✓ Successfully loaded 199160 rows


KeyError: 'admin_id'

In [ ]:
save_data(admin_group_df, "data/raw/customer_profile.parquet")

In [ ]:
@dataclass
class ValidationResult:
    """Stores validation results for a dataset."""
    is_valid: bool
    total_records: int
    valid_records: int
    invalid_records: int
    errors: List[Dict[str, Any]]
    warnings: List[Dict[str, Any]]
    quality_score: float
    
    def report(self) -> str:
        """Generate human-readable validation report."""
        report = f"""
{'='*70}
DATA VALIDATION REPORT
{'='*70}
Valid: {self.is_valid}
Quality Score: {self.quality_score:.2%}
Total Records: {self.total_records}
Valid Records: {self.valid_records}
Invalid Records: {self.invalid_records}

ERRORS ({len(self.errors)}):
{self._format_issues(self.errors)}

WARNINGS ({len(self.warnings)}):
{self._format_issues(self.warnings)}
{'='*70}
        """
        return report
    
    def _format_issues(self, issues: List[Dict]) -> str:
        """Format error/warning list."""
        if not issues:
            return "  None"
        return "\n".join([f"  - {issue['message']}" for issue in issues[:10]])


class DataValidator:
    """
    Validates data against schema configurations.
    
    Loads YAML schema files and applies comprehensive validation rules
    to pandas DataFrames including type checking, constraint validation,
    and data quality checks.
    """
    
    def __init__(self, schema_path: str):
        """
        Initialize validator with schema file.
        
        Args:
            schema_path (str): Path to schema YAML file
        """
        self.schema_path = Path(schema_path)
        self.schema_config = self._load_schema()
        self.columns_config = self.schema_config.get('columns', {})
        self.quality_rules = self.schema_config.get('quality_rules', {})

        logger.info(f"Schema loaded from {schema_path}")
    
    def _load_schema(self) -> Dict:
        """Load and parse YAML schema file."""
        try:
            with open(self.schema_path, 'r', encoding='utf-8') as f:
                schema = yaml.safe_load(f)
            schema_config = DataValidationSchemaConfig(**schema)
            logger.info(f"Successfully loaded schema: {self.schema_path}")
            return schema_config
        except FileNotFoundError:
            logger.error(f"Schema file not found: {self.schema_path}")
            raise
        except yaml.YAMLError as e:
            logger.error(f"Error parsing YAML schema: {e}")
            raise
    
    def validate(self, df: pd.DataFrame) -> ValidationResult:
        """
        Run all validation checks on DataFrame.
        
        Args:
            df (pd.DataFrame): Data to validate
        
        Returns:
            ValidationResult: Comprehensive validation results
        """
        errors = []
        warnings = []
        
        logger.info(f"Starting validation for {len(df)} records")
        
        # 1. Check required columns exist
        errors.extend(self._validate_columns_exist(df))
        
        # 2. Validate data types
        errors.extend(self._validate_data_types(df))
        
        # 3. Validate null/missing values
        warnings.extend(self._validate_nullability(df))
        
        # 4. Validate constraints (unique, foreign keys, etc.)
        errors.extend(self._validate_constraints(df))
        
        # 5. Validate business rules (enums, patterns, ranges)
        errors.extend(self._validate_business_rules(df))
        
        # 6. Detect anomalies
        warnings.extend(self._detect_anomalies(df))
        
        # Calculate quality score
        valid_records = len(df) - len([e for e in errors if 'row' in str(e)])
        quality_score = valid_records / len(df) if len(df) > 0 else 0
        
        is_valid = len(errors) == 0
        
        result = ValidationResult(
            is_valid=is_valid,
            total_records=len(df),
            valid_records=valid_records,
            invalid_records=len([e for e in errors if 'row' in str(e)]),
            errors=errors,
            warnings=warnings,
            quality_score=quality_score
        )
        
        logger.info(f"Validation complete. Quality Score: {quality_score:.2%}")
        return result
    
    def _validate_columns_exist(self, df: pd.DataFrame) -> List[Dict]:
        """Check all required columns exist in DataFrame."""
        errors = []
        required_columns = set(self.columns_config.keys())
        df_columns = set(df.columns)
        
        missing_columns = required_columns - df_columns
        if missing_columns:
            error = {
                'type': 'MissingColumn',
                'columns': list(missing_columns),
                'message': f"Missing required columns: {missing_columns}"
            }
            errors.append(error)
            logger.warning(f"Missing columns: {missing_columns}")
        
        extra_columns = df_columns - required_columns
        if extra_columns:
            logger.info(f"Extra columns found (will be ignored): {extra_columns}")
        
        return errors
    
    def _validate_data_types(self, df: pd.DataFrame) -> List[Dict]:
        """Validate column data types."""
        errors = []
        
        for column, config in self.columns_config.items():
            if column not in df.columns:
                continue
            
            expected_type = config.get('type')
            
            try:
                if expected_type == 'date':
                    # Try to parse as date
                    pd.to_datetime(df[column], errors='coerce')
                    null_count = df[column].isna().sum()
                    if null_count > 0 and not config.get('nullable', False):
                        errors.append({
                            'column': column,
                            'type': 'InvalidDateFormat',
                            'message': f"Column '{column}' has {null_count} invalid dates"
                        })
                
                elif expected_type == 'datetime':
                    pd.to_datetime(df[column], errors='coerce')
                    null_count = df[column].isna().sum()
                    if null_count > 0 and not config.get('nullable', False):
                        errors.append({
                            'column': column,
                            'type': 'InvalidDatetimeFormat',
                            'message': f"Column '{column}' has {null_count} invalid datetimes"
                        })
                
                elif expected_type == 'numeric':
                    pd.to_numeric(df[column], errors='coerce')
                    null_count = df[column].isna().sum()
                    if null_count > 0 and not config.get('nullable', False):
                        errors.append({
                            'column': column,
                            'type': 'InvalidNumeric',
                            'message': f"Column '{column}' has {null_count} non-numeric values"
                        })
            
            except Exception as e:
                logger.error(f"Error validating type for column '{column}': {e}")
        
        return errors
    
    def _validate_nullability(self, df: pd.DataFrame) -> List[Dict]:
        """Validate null/missing values against schema."""
        warnings = []
        null_tolerance = self.quality_rules.get('null_tolerance', {})
        
        for column, config in self.columns_config.items():
            if column not in df.columns:
                continue
            
            null_count = df[column].isna().sum()
            total_count = len(df)
            null_percentage = (null_count / total_count * 100) if total_count > 0 else 0
            
            is_nullable = config.get('nullable', False)
            tolerance = null_tolerance.get(column, 0)
            
            # Check if nulls exceed tolerance
            if null_percentage > tolerance:
                if not is_nullable and null_count > 0:
                    warnings.append({
                        'column': column,
                        'type': 'NullableViolation',
                        'null_percentage': null_percentage,
                        'message': f"Column '{column}' has {null_percentage:.2f}% null values "
                                   f"(tolerance: {tolerance}%)"
                    })
            
            logger.info(f"Column '{column}': {null_percentage:.2f}% null values")
        
        return warnings
    
    def _validate_constraints(self, df: pd.DataFrame) -> List[Dict]:
        """Validate primary key, unique, and foreign key constraints."""
        errors = []
        
        # Check unique constraints
        primary_keys = self.table_constraints.get('primary_keys', [])
        for column in primary_keys:
            if column not in df.columns:
                continue
            
            duplicate_count = df[column].duplicated().sum()
            if duplicate_count > 0:
                errors.append({
                    'column': column,
                    'type': 'DuplicateKeyViolation',
                    'duplicate_count': duplicate_count,
                    'message': f"Column '{column}' has {duplicate_count} duplicate values"
                })
                logger.error(f"Duplicate values found in '{column}'")
        
        # Check primary key
        primary_key = self.table_constraints.get('primary_key')
        if primary_key and primary_key in df.columns:
            duplicate_count = df[primary_key].duplicated().sum()
            null_count = df[primary_key].isna().sum()
            
            if duplicate_count > 0:
                errors.append({
                    'column': primary_key,
                    'type': 'PrimaryKeyDuplicate',
                    'duplicate_count': duplicate_count,
                    'message': f"Primary key '{primary_key}' has {duplicate_count} duplicates"
                })
            
            if null_count > 0:
                errors.append({
                    'column': primary_key,
                    'type': 'PrimaryKeyNull',
                    'null_count': null_count,
                    'message': f"Primary key '{primary_key}' has {null_count} null values"
                })
        
        return errors
    
    def _validate_business_rules(self, df: pd.DataFrame) -> List[Dict]:
        """Validate business rules (enums, patterns, ranges)."""
        errors = []
        
        for column, config in self.columns_config.items():
            if column not in df.columns:
                continue
            
            constraints = config.get('constraints', [])
            
            for constraint in constraints:
                constraint_type = constraint.get('type')
                
                # Enum constraint
                if constraint_type == 'enum':
                    allowed_values = constraint.get('values', [])
                    invalid_rows = ~df[column].isin(allowed_values) & df[column].notna()
                    invalid_count = invalid_rows.sum()
                    
                    if invalid_count > 0:
                        errors.append({
                            'column': column,
                            'type': 'EnumViolation',
                            'allowed_values': allowed_values,
                            'invalid_count': invalid_count,
                            'message': f"Column '{column}' has {invalid_count} values outside "
                                       f"allowed enum: {allowed_values}"
                        })
                        logger.warning(f"Enum violation in '{column}'")
                
                # Pattern constraint (regex)
                elif constraint_type == 'pattern':
                    pattern = constraint.get('value')
                    invalid_rows = df[column].astype(str).str.match(pattern) == False
                    invalid_count = invalid_rows.sum()
                    
                    if invalid_count > 0:
                        errors.append({
                            'column': column,
                            'type': 'PatternViolation',
                            'pattern': pattern,
                            'invalid_count': invalid_count,
                            'message': f"Column '{column}' has {invalid_count} values not matching "
                                       f"pattern: {pattern}"
                        })
                        logger.warning(f"Pattern violation in '{column}'")
                
                # Date range constraint
                elif constraint_type == 'date_range':
                    min_date = constraint.get('min')
                    max_date = constraint.get('max')
                    
                    try:
                        df_dates = pd.to_datetime(df[column], errors='coerce')
                        if min_date:
                            before_min = (df_dates < pd.to_datetime(min_date)).sum()
                            if before_min > 0:
                                errors.append({
                                    'column': column,
                                    'type': 'DateRangeViolation',
                                    'min_date': min_date,
                                    'violation_count': before_min,
                                    'message': f"Column '{column}' has {before_min} dates before {min_date}"
                                })
                        
                        if max_date:
                            after_max = (df_dates > pd.to_datetime(max_date)).sum()
                            if after_max > 0:
                                errors.append({
                                    'column': column,
                                    'type': 'DateRangeViolation',
                                    'max_date': max_date,
                                    'violation_count': after_max,
                                    'message': f"Column '{column}' has {after_max} dates after {max_date}"
                                })
                    except Exception as e:
                        logger.error(f"Error validating date range for '{column}': {e}")
        
        return errors
    
    def _detect_anomalies(self, df: pd.DataFrame) -> List[Dict]:
        """Detect statistical anomalies."""
        warnings = []
        statistical_bounds = self.quality_rules.get('statistical_bounds', {})
        
        if not statistical_bounds.get('enabled', False):
            return warnings
        
        for column, config in self.columns_config.items():
            if column not in df.columns:
                continue
            
            # Check for not_future constraint on dates
            constraints = config.get('constraints', [])
            for constraint in constraints:
                if constraint.get('type') == 'not_future':
                    try:
                        df_dates = pd.to_datetime(df[column], errors='coerce')
                        future_count = (df_dates > datetime.now()).sum()
                        
                        if future_count > 0:
                            warnings.append({
                                'column': column,
                                'type': 'FutureDate',
                                'anomaly_count': future_count,
                                'message': f"Column '{column}' has {future_count} future dates"
                            })
                            logger.warning(f"Future dates detected in '{column}'")
                    except Exception as e:
                        logger.error(f"Error checking future dates in '{column}': {e}")
        
        return warnings

In [4]:
sample_data = {
    'user_id': ['2', '2', np.nan, '4', '4'],
    'first_name': ['John', 'Jane', 'Bob' ,'Alice', 'Alice'],
    'last_name': ['Doe', 'Smith', 'Johnson', 'Brown', 'Brown'],
    'activated': ['True', 'True', 'False', 'True', 'True'],
    'admin_id': ['0.1', '0', '0', '3', '3'],
    'sex': ['male', 'female', 'male', 'female', 'female'],
    'foreigner': ['0', '0', '0', '0', '0'],
    'birthdate': ['dfs', '1985-03-22', '1992-07-10', '1990-12-01', '1990-12-01'],
    'registed_time': ['2023-01-01 10:30:00', '2023-01-05 14:20:00', '2023-01-10 09:15:00', '2023-01-12 11:00:00', '2023-01-12 11:00:00']
}

df = pd.DataFrame(sample_data)
df = generate_sk_key(df)

In [ ]:
class DataValidator:
    """
    Validates data against schema configurations.
    
    Loads YAML schema files and applies comprehensive validation rules
    to pandas DataFrames including type checking, constraint validation,
    and data quality checks.
    """
    
    def __init__(self, schema_path: str):
        """
        Initialize validator with schema file.
        
        Args:
            schema_path (str): Path to schema YAML file
        """
        self.schema_path = Path(schema_path)
        self.schema_config = load_single_config(DataValidationSchemaConfig, self.schema_path)
        self.columns_config = self.schema_config.columns
        self.quality_rules_config = self.schema_config.quality_rules

        logger.info(f"Schema loaded from {schema_path}")
    

    def _validate_columns_exist(self, df: pd.DataFrame) -> List[Dict]:
        """
        Check all required columns exist in DataFrame.
        
        Args:
            df (pd.DataFrame): The DataFrame to validate.

        Returns:
            List[Dict]: A list of error dictionaries for missing columns.
        """
        errors = []
        required_columns = set(self.columns_config.keys())
        df_columns = set(df.columns)
        
        missing_columns = required_columns - df_columns
        if missing_columns:
            error_df = df.copy()
            error_df["error_type"] = "MissingColumn"
            error_df["error_message"] = f"Missing required columns: {missing_columns}"
            error = {
                'type': 'MissingColumn',
                'columns': list(missing_columns),
                'message': f"Missing required columns: {missing_columns}",
                'error_df': error_df
            }
            errors.append(error)
            logger.warning(f"Missing columns: {missing_columns}")
        
        extra_columns = df_columns - required_columns
        if extra_columns:
            logger.info(f"Extra columns found (will be ignored): {extra_columns}")
        
        return errors


    def _validate_record_duplicates(self, df: pd.DataFrame) -> List[Dict]:
        """
        Check for duplicate records in the DataFrame.

        Args:
            df (pd.DataFrame): The DataFrame to validate.

        Returns:
            List[Dict]: A list of error dictionaries for duplicate records.
        """
        errors = []
        check_df = df.copy().drop("sk_key", axis=1, errors='ignore')
        duplicate_mask = check_df.duplicated(keep=False)
        error_df = df[duplicate_mask].copy()

        if not error_df.empty:
            error_df["error_type"] = "RecordDuplicateViolation"
            error_df["error_message"] = f"Duplicate records found based on all columns except"
            errors.append({
                'type': 'RecordDuplicateViolation',
                'message': f"Duplicate records found based on all columns except 'sk_key'",
                'error_df': error_df
            })

        return errors


    def _validate_unique_key_duplicates(self, df: pd.DataFrame) -> List[Dict]:
        """
        Check for duplicate records based on unique key columns.

        Args:
            df (pd.DataFrame): The DataFrame to validate.

        Returns:
            List[Dict]: A list of error dictionaries for unique key duplicates.
        """
        errors = []
        check_df = df.copy()
        key_columns = [column for column, config in self.columns_config.items() if config.primary_keys]
        duplicate_mask = check_df[key_columns].duplicated(keep=False)
        error_df = df[duplicate_mask].copy()

        if not error_df.empty:
            error_df["error_type"] = "UniqueKeyDuplicateViolation"
            error_df["error_message"] = f"Duplicate values found in unique key column: {key_columns}"
            errors.append({
                'type': 'UniqueKeyDuplicateViolation',
                'message': f"Duplicate values found in unique key column: {key_columns}",
                'error_df': error_df
            })

        return errors

    
    def _validate_data_types(self, df: pd.DataFrame) -> List[Dict]:
        """
        Validate column data types.

        Args:
            df (pd.DataFrame): The DataFrame to validate.

        Returns:
            List[Dict]: A list of error dictionaries for invalid data types.
        """
        def _validate_datetime_column(df: pd.DataFrame, column: str, expected_type: str) -> Optional[Dict]:
            """
            Validate datetime/date column.

            Args:
                df (pd.DataFrame): The DataFrame to validate.
                column (str): The name of the column to validate.
                expected_type (str): The expected data type of the column.

            Returns:
                Optional[Dict]: An error dictionary if validation fails, None otherwise.
            """
            check_df = df.copy()
            parsed_col = f"{column}_parsed"
            check_df[parsed_col] = pd.to_datetime(check_df[column], errors='coerce')

            error_mask = check_df[column].notna() & check_df[parsed_col].isna()
            if not error_mask.any():
                return None

            error_df = check_df[error_mask].drop(columns=[parsed_col])
            error_df["error_type"] = "InvalidDataType"
            error_df["error_message"] = f"Invalid column type '{expected_type}': {column}"

            return {
                'column': column,
                'type': f"Invalid{expected_type.capitalize()}Format",
                'message': f"Column '{column}' has {error_df.shape[0]} invalid {expected_type} values",
                'error_df': error_df
            }


        def _validate_numeric_column(df: pd.DataFrame, column: str, expected_type: str) -> Optional[Dict]:
            """
            Validate numeric column.
            
            Args:
                df (pd.DataFrame): The DataFrame to validate.
                column (str): The name of the column to validate.
                expected_type (str): The expected data type of the column.

            Returns:
                Optional[Dict]: An error dictionary if validation fails, None otherwise.
            """

            check_df = df.copy()
            parsed_col = f"{column}_parsed"
            check_df[parsed_col] = pd.to_numeric(check_df[column], errors='coerce')

            error_mask = check_df[column].notna() & check_df[parsed_col].isna()
            if not error_mask.any():
                return None

            error_df = check_df[error_mask].drop(columns=[parsed_col])
            error_df["error_type"] = "InvalidDataType"
            error_df["error_message"] = f"Invalid column type 'numeric': {column}"

            return {
                'column': column,
                'type': "InvalidNumeric",
                'message': f"Column '{column}' has {error_df.shape[0]} non-numeric values",
                'error_df': error_df
            }


        def _validate_bool_column(df: pd.DataFrame, column: str, expected_type: str) -> Optional[Dict]:
            """
            Validate boolean column.

            Args:
                df (pd.DataFrame): The DataFrame to validate.
                column (str): The name of the column to validate.
                expected_type (str): The expected data type of the column.

            Returns:
                Optional[Dict]: An error dictionary if validation fails, None otherwise.
            """
            check_df = df.copy()
            valid_values = {True, False, 'True', 'False', 1, 0}
            invalid_mask = ~check_df[column].isin(valid_values) & check_df[column].notna()

            if not invalid_mask.any():
                return None

            error_df = check_df[invalid_mask].copy()
            error_df["error_type"] = "InvalidDataType"
            error_df["error_message"] = f"Invalid column type 'bool': {column}"

            return {
                'column': column,
                'type': "InvalidBoolean",
                'message': f"Column '{column}' has {error_df.shape[0]} non-boolean values",
                'error_df': error_df
            }

        errors = []
        # Helper mapping between expected type and validator function
        type_validators = {
            'date': _validate_datetime_column,
            'datetime': _validate_datetime_column,
            'integer': _validate_numeric_column,
            'float': _validate_numeric_column,
            'bool': _validate_bool_column,
        }
        
        for column, config in self.columns_config.items():
            if column not in df.columns:
                continue
            
            expected_type = config.type.lower().strip()
            if expected_type == 'string':
                continue

            validator = type_validators.get(expected_type)

            if not validator:
                logger.warning(f"No validator defined for column type '{expected_type}' ({column})")
                continue

            try:
                error = validator(df, column, expected_type)
                if error:
                    errors.append(error)
            except Exception as e:
                logger.error(f"Error validating type for column '{column}': {e}")

        return errors


    def _validate_nullability(self, df: pd.DataFrame) -> List[Dict]:
        """
        Validate null/missing values against schema.

        Args:
            df (pd.DataFrame): The DataFrame to validate.
        
        Returns:
            List[Dict]: A list of warning dictionaries for nullability violations.
        """
        errors = []
        for column, config in self.columns_config.items():
            if column not in df.columns:
                continue

            is_nullable = config.nullable
            null_count = df[column].isna().sum()
            
            # Check if nulls 
            if not is_nullable and null_count > 0:
                error_df = df[df[column].isna()].copy()
                error_df["error_type"] = "NullableViolation"
                error_df["error_message"] = f"Column '{column}' has null values"
                errors.append({
                    'column': column,
                    'type': 'NullableViolation',
                    'message': f"Column '{column}' has null values",
                    'error_df': error_df
                })
            
            logger.info(f"Column '{column}': has null values")
        
        return errors


    def _validate_constraints(self, df: pd.DataFrame) -> List[Dict]:
        """Validate business rules (enums, patterns, ranges)."""
        errors = []
        
        for column, config in self.columns_config.items():
            if column not in df.columns:
                continue
            
            constraints = config.get('constraints', [])
            
            for constraint in constraints:
                constraint_type = constraint.get('type')
                
                # Enum constraint
                if constraint_type == 'enum':
                    allowed_values = constraint.get('values', [])
                    invalid_rows = ~df[column].isin(allowed_values) & df[column].notna()
                    invalid_count = invalid_rows.sum()
                    
                    if invalid_count > 0:
                        errors.append({
                            'column': column,
                            'type': 'EnumViolation',
                            'allowed_values': allowed_values,
                            'invalid_count': invalid_count,
                            'message': f"Column '{column}' has {invalid_count} values outside "
                                       f"allowed enum: {allowed_values}"
                        })
                        logger.warning(f"Enum violation in '{column}'")
                
                # Pattern constraint (regex)
                elif constraint_type == 'pattern':
                    pattern = constraint.get('value')
                    invalid_rows = df[column].astype(str).str.match(pattern) == False
                    invalid_count = invalid_rows.sum()
                    
                    if invalid_count > 0:
                        errors.append({
                            'column': column,
                            'type': 'PatternViolation',
                            'pattern': pattern,
                            'invalid_count': invalid_count,
                            'message': f"Column '{column}' has {invalid_count} values not matching "
                                       f"pattern: {pattern}"
                        })
                        logger.warning(f"Pattern violation in '{column}'")
                
                # Date range constraint
                elif constraint_type == 'date_range':
                    min_date = constraint.get('min')
                    max_date = constraint.get('max')
                    
                    try:
                        df_dates = pd.to_datetime(df[column], errors='coerce')
                        if min_date:
                            before_min = (df_dates < pd.to_datetime(min_date)).sum()
                            if before_min > 0:
                                errors.append({
                                    'column': column,
                                    'type': 'DateRangeViolation',
                                    'min_date': min_date,
                                    'violation_count': before_min,
                                    'message': f"Column '{column}' has {before_min} dates before {min_date}"
                                })
                        
                        if max_date:
                            after_max = (df_dates > pd.to_datetime(max_date)).sum()
                            if after_max > 0:
                                errors.append({
                                    'column': column,
                                    'type': 'DateRangeViolation',
                                    'max_date': max_date,
                                    'violation_count': after_max,
                                    'message': f"Column '{column}' has {after_max} dates after {max_date}"
                                })
                    except Exception as e:
                        logger.error(f"Error validating date range for '{column}': {e}")
        
        return errors


    def validate(self, df: pd.DataFrame) -> ValidationResult:


        # allow_record_duplicates = self.quality_rules_config.allow_record_duplicates.enabled
        # if not allow_record_duplicates:

In [ ]:
try:
    logger.info("Starting data validation...")

    all_errors = []

    # Load config
    pipeline_config = load_single_config(PipelineConfig, config_path)
    schema_path = pipeline_config.validation.config_path
    
    validator = DataValidator(schema_path)
    input_path = pipeline_config.validation.input_data_path
    output_path = pipeline_config.validation.output_data_path
    
    input_df = load_data(source=input_path)
    

except Exception as e:
    logger.exception(f"Error during data validation: {e}")
    raise e

In [36]:
import pandera.pandas as pa
from pandera import Column, DataFrameSchema

class DataValidator:
    """
    Validates data against schema configurations.
    
    Loads YAML schema files and applies comprehensive validation rules
    to pandas DataFrames including type checking, constraint validation,
    and data quality checks.
    """
    
    def __init__(self, config_path: str):
        """
        Initialize validator with config file.
        
        Args:
            config_path (str): Path to config YAML file
        """
        try:
            logger.info("Initializing DataValidator...")
            # Load config
            self.config_path = Path(config_path)
            self.pipeline_config = load_single_config(PipelineConfig, self.config_path)

            self.schema_path = Path(self.pipeline_config.validation.config_path)
            self.schema_config = load_single_config(DataValidationConfig, self.schema_path)
            
            self.columns_config = self.schema_config.columns
            self.quality_rules_config = self.schema_config.quality_rules

            self.input_path = self.pipeline_config.validation.input.get("file_path")
            self.output_path = self.pipeline_config.validation.output.get("file_path")

        except Exception as e:
            logger.error(f"Error initializing DataValidator: {e}")
            raise e


    def build_pandera_schema(self) -> pa.DataFrameSchema:
        """
        Dynamically build Pandera DataFrameSchema based on columns_config.
        """
        columns = {}

        type_map = {
            'string': pa.String,
            'integer': pa.Int,
            'float': pa.Float,
            'bool': pa.Bool,
            'date': pa.DateTime,
            'datetime': pa.DateTime,
        }

        for col_name, config in self.columns_config.items():
            pandas_dtype = type_map.get(config.type.lower().strip())
            columns[col_name] = Column(
                dtype=pandas_dtype,
                nullable=config.nullable,
            )

        schema = DataFrameSchema(columns, coerce=True, strict=True)
        return schema
    

    def validate_duplicates_data(self, df: pd.DataFrame) -> dict:
        """
        Add duplicate record and unique key checks.
        """
        result = {}
        errors = []

        # Check duplicate rows (excluding 'sk_key')
        if df.duplicated(subset=[c for c in df.columns if c != 'sk_key']).any():
            duplicate_df = df[df.duplicated(subset=[c for c in df.columns if c != 'sk_key'], keep='last')]
            duplicate_df.loc['error_type'] = "RecordDuplicateViolation"
            duplicate_df.loc['error_message'] = "Duplicate records found based on all columns except 'sk_key'"
            errors.append({
                'type': 'RecordDuplicateViolation',
                'message': f"Duplicate records found based on all columns except 'sk_key'",
                'error_df': duplicate_df
            })

        clean_df = df[~ df.duplicated(subset=[c for c in df.columns if c != 'sk_key'], keep='last')]
        
        # Check unique key constraint
        key_columns = [c for c, cfg in self.columns_config.items() if getattr(cfg, 'primary_keys', True)]
        if key_columns and clean_df.duplicated(subset=key_columns).any():
            duplicate_df = clean_df[clean_df.duplicated(subset=key_columns, keep=False)]
            duplicate_df['error_type'] = "UniqueKeyDuplicateViolation"
            duplicate_df['error_message'] = f"Duplicate values found in unique key column(s): {key_columns}"
            errors.append({
                'type': 'UniqueKeyDuplicateViolation',
                'message': f"Duplicate values found in unique key column(s): {key_columns}",
                'error_df': duplicate_df
            })

        clean_df = clean_df[~clean_df.duplicated(subset=key_columns, keep=False)]
        
        result['error'] = errors
        result['clean_df'] = clean_df
        
        return result


    def validate_data_types_and_nullability(self, df: pd.DataFrame) -> dict:
        """
        Validate data types and nullability using Pandera.

        Args:
            df (pd.DataFrame): The DataFrame to validate.

        Returns:
            dict: A dictionary with 'error' and 'clean_df' keys.
        """
        result = {}
        errors = []

        # Build schema dynamically
        schema = self.build_pandera_schema()

        try:
            schema.validate(df, lazy=True)
        except pa.errors.SchemaErrors as err:
            for failure in err.failure_cases.to_dict(orient="records"):
                check = failure.get("check")
                failure_case = failure.get("failure_case")
                index = failure.get("index")

                if (check.startswith("coerce_dtype") and pd.isna(failure_case)) or not index:
                    continue # skip data type errors due to missing data

                if pd.isna(failure_case):
                    error_df = df.loc[[index]].copy()
                    error_df["error_type"] = "NullableViolation"
                    error_df["error_message"] = f"Column '{failure.get('column')}' is non-nullable but has null value"
                    error = {
                        'column': failure.get("column"),
                        'type': "NullableViolation",
                        'message': failure_case,
                        'check': check,
                        'index': index,
                        'error_df': error_df
                    }
                
                elif check.startswith("coerce_dtype"):
                    error_df = df.loc[[index]].copy()
                    error_df["error_type"] = "InvalidDataType"
                    error_df["error_message"] = f"Column '{failure.get('column')}' has invalid data type"
                    error = {
                        'column': failure.get("column"),
                        'type': "InvalidDataType",
                        'message': failure_case,
                        'check': check,
                        'index': index,
                        'error_df': error_df
                    }
                errors.append(error)

        errors_indices = {error.get('index') for error in errors}
        clean_df = df[~df.index.isin(errors_indices)]

        result['error'] = errors
        result['clean_df'] = clean_df

        return result
    

    def validate_foreign_keys(self, df: pd.DataFrame) -> dict:
        """
        Validate foreign key constraints.

        Args:
            df (pd.DataFrame): The DataFrame to validate.

        Returns:
            dict: A dictionary with 'error' and 'clean_df' keys.
        """
        result = {}
        errors = []

        for column, config in validator.columns_config.items():
            foreign_key = config.__dict__.get("foreign_keys", None)
            if not foreign_key:
                continue
            
            child_path = foreign_key.child_path
            child_column = foreign_key.child_column

            # Load child table data
            child_df = load_data(source=child_path)
            valid_keys = set(child_df[child_column].dropna().unique())

            invalid_mask = ~df[column].isin(valid_keys) & df[column].notna()
            if invalid_mask.any():
                error_df = df[invalid_mask].copy()
                error_df["error_type"] = "ForeignKeyViolation"
                error_df["error_message"] = f"Column '{column}' has values not present in '{child_path}.{child_column}'"
                errors.append({
                    'column': column,
                    'type': 'ForeignKeyViolation',
                    'message': f"Column '{column}' has values not present in '{child_path}.{child_column}'",
                    'error_df': error_df
                })
                clean_df = df[~invalid_mask]

        result['error'] = errors
        result['clean_df'] = clean_df

        return result
    
    # def validator(self, df: pd.DataFrame) -> ValidationResult:
    #     """
    #     Validate DataFrame against schema rules.

    #     Args:
    #         df (pd.DataFrame): The DataFrame to validate.

    #     Returns:
    #         ValidationResult: The result of the validation.
    #     """
    #     all_errors = []

    #     # Validate missing columns
    #     errors = self._validate_columns_exist(df)
    #     all_errors.extend(errors)

    #     # Validate duplicate records
    #     dup_result = self.validate_duplicates_data(df)
    #     all_errors.extend(dup_result['error'])
    #     df = dup_result['clean_df']

    #     # Validate data types and nullability
    #     type_null_result = self.validate_data_types_and_nullability(df)
    #     all_errors.extend(type_null_result['error'])
    #     df = type_null_result['clean_df']

    #     # Validate foreign keys
    #     fk_result = self.validate_foreign_keys(df)
    #     all_errors.extend(fk_result['error'])
    #     df = fk_result['clean_df']

    #     # Validate primary key, unique, and foreign key constraints
    #     constraint_errors = self._validate_constraints(df)
    #     all_errors.extend(constraint_errors)

    #     # Validate business rules
    #     business_rule_errors = self._validate_business_rules(df)
    #     all_errors.extend(business_rule_errors)

    #     # Detect anomalies
    #     anomaly_warnings = self._detect_anomalies(df)
    #     all_errors.extend(anomaly_warnings)

    #     return ValidationResult(errors=all_errors, clean_df=df)

In [27]:
# input parameters
config_path = "src/churn_prediction/config/conf/conf_customer_profile.yaml"

In [37]:
try:
    logger.info("Starting data validation...")

    all_errors = []

    validator = DataValidator(config_path)

    input_df = load_data(source=validator.input_path)
    

except Exception as e:
    logger.exception(f"Error during data validation: {e}")
    raise e


# # Validate missing columns
# errors = self._validate_columns_exist(df)
# all_errors.extend(errors)

# # Validate duplicate records
# dup_result = self.validate_duplicates_data(df)
# all_errors.extend(dup_result['error'])
# df = dup_result['clean_df']

# # Validate data types and nullability
# type_null_result = self.validate_data_types_and_nullability(df)
# all_errors.extend(type_null_result['error'])
# df = type_null_result['clean_df']

# # Validate foreign keys
# fk_result = self.validate_foreign_keys(df)
# all_errors.extend(fk_result['error'])
# df = fk_result['clean_df']

# # Validate primary key, unique, and foreign key constraints
# constraint_errors = self._validate_constraints(df)
# all_errors.extend(constraint_errors)

# # Validate business rules
# business_rule_errors = self._validate_business_rules(df)
# all_errors.extend(business_rule_errors)

# # Detect anomalies
# anomaly_warnings = self._detect_anomalies(df)
# all_errors.extend(anomaly_warnings)

[ 2025-11-08 12:57:53 ] | churn_prediction | INFO     | 1770599734.py:<module>:2 | Starting data validation...
[ 2025-11-08 12:57:53 ] | churn_prediction | INFO     | 1408433689.py:__init__:21 | Initializing DataValidator...
[ 2025-11-08 12:57:53 ] | churn_prediction | INFO     | common.py:load_single_config:36 | Successfully loaded schema: src/churn_prediction/config/conf/conf_customer_profile.yaml
[ 2025-11-08 12:57:53 ] | churn_prediction | INFO     | common.py:load_single_config:36 | Successfully loaded schema: src/churn_prediction/config/data_validation/customer_profile.yaml
[ 2025-11-08 12:57:53 ] | churn_prediction | INFO     | loaders.py:load_data:343 | Auto-detecting format for local file: data/raw/customer_profile.parquet
[ 2025-11-08 12:57:53 ] | churn_prediction | WARNING  | loaders.py:ensure_file_path:218 | File not found: data/raw/customer_profile.parquet
[ 2025-11-08 12:57:53 ] | churn_prediction | ERROR    | loaders.py:load_parquet:269 | ✗ Failed to load Parquet: File n

FileNotFoundError: File not found: data/raw/customer_profile.parquet

In [34]:
validator

In [21]:
pipeline_config.validation.config_path

'src/churn_prediction/config/data_validation/customer_profile.yaml'

In [ ]:
clean_df = validator.validate_duplicates_data(df)



/var/folders/cj/gfy7l4ys0m7gd6jgx2hnycl40000gn/T/ipykernel_91574/3737464937.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  duplicate_df.loc['error_type'] = "RecordDuplicateViolation"
/var/folders/cj/gfy7l4ys0m7gd6jgx2hnycl40000gn/T/ipykernel_91574/3737464937.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  duplicate_df.loc['error_message'] = "Duplicate records found based on all columns except 'sk_key'"
/var/folders/cj/gfy7l4ys0m7gd6jgx2hnycl40000gn/T/ipykernel_91574/3737464937.py:79: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_ind

In [10]:
clean_df = clean_df['clean_df']

In [11]:
clean_df

,user_id,first_name,last_name,activated,admin_id,sex,foreigner,birthdate,registed_time,sk_key
2,NaN,Bob,Johnson,False,0,male,0,1992-07-10,2023-01-10 09:15:00,3
4,4,Alice,Brown,True,3,female,0,1990-12-01,2023-01-12 11:00:00,5


In [12]:
clean_df = validator.validate_data_types_and_nullability(clean_df)

/Users/supawitjunsiritrakhoon/Desktop/Customer_Churn_Prediction/Project_file/customer-churn-prediction/venv/lib/python3.12/site-packages/pandera/_pandas_deprecated.py:149: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


In [13]:
clean_df = clean_df['clean_df']

In [14]:
clean_df

,user_id,first_name,last_name,activated,admin_id,sex,foreigner,birthdate,registed_time,sk_key
4,4,Alice,Brown,True,3,female,0,1990-12-01,2023-01-12 11:00:00,5
